In [159]:
#importing pandas
import pandas as pd
import numpy as np

In [160]:
#loading the datasets
df_2019 = pd.read_csv("data/2019_LoL_esports_match_data_from_OraclesElixir.csv")
df_2022 = pd.read_csv("data/2022_LoL_esports_match_data_from_OraclesElixir.csv")
df_2025 = pd.read_csv("data/2025_LoL_esports_match_data_from_OraclesElixir.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'data/2019_LoL_esports_match_data_from_OraclesElixir.csv'

In [ ]:


cols = [
    "year", "league", "teamname", "result", "position", "side",
    "goldat15", "xpat15", "csat15",
    "opp_goldat15", "opp_xpat15", "opp_csat15",
    "golddiffat15", "xpdiffat15", "csdiffat15",
    "killsat15", "assistsat15", "deathsat15",
    "opp_killsat15", "opp_assistsat15", "opp_deathsat15",
    "gamelength", "patch", "date",
    "firstherald", "firstbaron", "firsttower",
    "firstmidtower", "firsttothreetowers",
    "barons", "heralds", "inhibitors",
    "firstblood", "firstdragon",
    "dragons", "elders"
]

df_2019_clean = df_2019.reindex(columns=cols).copy()
df_2022_clean = df_2022.reindex(columns=cols).copy()
df_2025_clean = df_2025.reindex(columns=cols).copy()

df = pd.concat([df_2019_clean, df_2022_clean, df_2025_clean], ignore_index=True)

# Filter to team rows ONLY
df = df[df["position"] == "team"]

# Keep only 2019, 2022, 2025
df = df[df["year"].isin([2019, 2022, 2025])]

# ONLY drop rows missing the absolute essentials — not all objective columns
df = df.dropna(subset=["result", "league", "year", "golddiffat15"])

print("Years present:")
print(df["year"].value_counts().sort_index())
print(f"\nTotal team rows: {len(df)}")

In [ ]:
# Recreate every derived column in one place , SAME AS ABOVE CELL!
df["got_inhibitor"] = (df["inhibitors"] >= 1).astype(int)
df["gold_lead"] = (df["golddiffat15"] > 0).astype(int)
df["xp_lead"] = (df["xpdiffat15"] > 0).astype(int)
df["dragon_soul"] = (df["dragons"] >= 4).astype(int)

bins = [-np.inf, -3000, -1000, 0, 1000, 3000, np.inf]
labels = ["big deficit", "deficit", "even", "small lead", "lead", "big lead"]
df["gold_bucket"] = pd.cut(df["golddiffat15"], bins=bins, labels=labels)

length_bins = [0, 1500, 1800, 2100, 2400, np.inf]
length_labels = ["<25min", "25-30min", "30-35min", "35-40min", "40min+"]
df["length_bucket"] = pd.cut(df["gamelength"], bins=length_bins, labels=length_labels)

df_main = df[df["league"].isin(["LCK", "LPL", "LEC", "LCS", "PCS"])]
comebacks_main = df_main[(df_main["golddiffat15"] > 1000) & (df_main["result"] == 0)]

In [ ]:
#exploring the dataset
early_cols = [c for c in df_2019.columns if "diffat" in c or "at15" in c or "at25" in c]
early_cols

In [ ]:
#exploring the dataset
early_cols = [c for c in df_2022.columns if "diffat" in c or "at15" in c or "at25" in c]
early_cols

In [ ]:
#exploring the dataset
early_cols = [c for c in df_2025.columns if "diffat" in c or "at15" in c or "at25" in c]
early_cols

In [ ]:
#exploring the data
set(df_2019.columns)
set(df_2022.columns)
set(df_2025.columns)

In [ ]:
#existence check 
[col for col in ["golddiffat15", "xpdiffat15", "csdiffat15", "result", "year", "league"] if col in df.columns]

In [ ]:
#overall correlation between early - game stats
# gold difference is the bigger predictor FOR NOW at 0.35.6 => 0.36
df[["golddiffat15", "xpdiffat15", "csdiffat15", "result"]].corr()["result"].sort_values(ascending=False)

In [ ]:
#year by year correlations , show stability across seasons
#this is how we adress the hypothesis, "does importance change across seasons
for y in df["year"].unique():
    sub = df[df["year"] == y]
    print(f"\nYear {y}")
    print(sub[["golddiffat15", "xpdiffat15", "csdiffat15", "result"]].corr()["result"])

In [ ]:
#Regional correlations which show PCS is the most consistent and LCS most volatile
# This adresses the hypothesis " do regions show different patterns? 
main_regions = ["LCK", "LPL", "LEC", "LCS", "PCS"]
df_main = df[df["league"].isin(main_regions)]

for region in df_main["league"].unique():
    sub = df_main[df_main["league"] == region]
    print(f"\nRegion: {region}")
    print(sub[["golddiffat15", "xpdiffat15", "csdiffat15", "result"]].corr()["result"])

In [ ]:
# This is where we check the comebacks
#Comeback means when a team had 1000+ gold lead at 15 minutes but lost the match
comebacks = df[(df["golddiffat15"] > 1000) & (df["result"] == 0)]
len(comebacks)

In [ ]:
#Here we filter comebacks to the 5 main regions for cleaner regional comparison
main_regions = ["LCK", "LPL", "LEC", "LCS", "PCS"]
df_main = df[df["league"].isin(main_regions)]
comebacks_main = comebacks[comebacks["league"].isin(main_regions)]
len(comebacks_main)
comebacks_main["league"].value_counts()

In [ ]:
#Comeback rate by region > we see that LPL teams throw(throw away their leads) most of their games , PCS throw the least.
total_games_by_region = df_main["league"].value_counts()
comeback_rate = comebacks_main["league"].value_counts() / total_games_by_region
comeback_rate

In [ ]:
#we see the comeback frequency by year. 
comebacks_by_year = comebacks_main["year"].value_counts().sort_index()
comebacks_by_year

In [ ]:
#Bucketing comebacks by lead size , most comebacks happen at 1k-2k leads 
import numpy as np

bins = [1000, 2000, 3000, 4000, 5000, np.inf]
labels = ["1k–2k", "2k–3k", "3k–4k", "4k–5k", "5k+"]

comebacks_main["lead_bucket"] = pd.cut(comebacks_main["golddiffat15"], bins=bins, labels=labels)

comebacks_main["lead_bucket"].value_counts().sort_index()

In [ ]:
#lead distribution by region 
comebacks_main.groupby("league")["lead_bucket"].value_counts().sort_index()

In [ ]:
#statistical summary of gold leads that got thrown - average = 1638
comebacks_main["golddiffat15"].describe()

In [ ]:
#Average thrown lead by region , LCS throws biggest leads on average > why this happens?
comebacks_main.groupby("league")["golddiffat15"].mean()

In [ ]:
#The length of the games that the comebacks happen - average 35min
comebacks_main["gamelength"].describe()

In [ ]:
#Game length stats for ALL games , aver - 32.7 min
#Comebacks happen in longer games, this means if a team wants to comeback they have to stretch the game
#Time = losing team comeback factor? 
df_main["gamelength"].describe()

In [ ]:
#Average comeback game length by region
comebacks_main.groupby("league")["gamelength"].mean()

In [ ]:
#winrate per major objective - baron and inhibitor are decisive - herald good statistic but not a priority!
objectives = ["firstherald", "firstbaron", "got_inhibitor"]

for obj in objectives:
    wr = df.groupby(obj)["result"].mean()
    print(f"\n{obj}:\n{wr}")

In [ ]:
# Objective win rates broken down by region and by year  
for obj in ["firstherald", "firstbaron", "got_inhibitor"]:
    print(f"\n=== {obj} win rate by league ===")
    print(df_main.groupby(["league", obj])["result"].mean().unstack())

# By year
for obj in ["firstherald", "firstbaron", "got_inhibitor"]:
    print(f"\n=== {obj} win rate by year ===")
    print(df.groupby(["year", obj])["result"].mean().unstack())

In [ ]:
#first link in casual chain > gold lead doubles the baron rate (62.5%>27.9%)
df["gold_lead"] = (df["golddiffat15"] > 0).astype(int)
df.groupby("gold_lead")["firstbaron"].mean()

In [ ]:
#gold buckets that show that baron rate and win rate scale together with gold lead
bins = [-np.inf, -3000, -1000, 0, 1000, 3000, np.inf]
labels = ["big deficit", "deficit", "even", "small lead", "lead", "big lead"]
df["gold_bucket"] = pd.cut(df["golddiffat15"], bins=bins, labels=labels)

df.groupby("gold_bucket")[["firstbaron", "result"]].mean()

In [ ]:
#The single most important finding
#Gold lead alone wins 0.44% of the game BUT gold lead WITH baron wins 0.918% of the games 
#Baron is the conversion mechanism , not just a side-effect
df.groupby(["gold_lead", "firstbaron"])["result"].mean().unstack()

In [ ]:
# Teams ahead in gold at 15
ahead = df[df["golddiffat15"] > 1000]

print("Win rate when ahead AND got baron:")
print(ahead[ahead["firstbaron"] == 1]["result"].mean())

print("Win rate when ahead but NO baron:")
print(ahead[ahead["firstbaron"] == 0]["result"].mean())

In [ ]:
# More gold at 15 = more kills (proxy for winning fights)
df[["golddiffat15", "killsat15"]].corr()

In [ ]:
#Ranking ALL early indicators by individual winrate
#Tower - based objectives outperform first blood and first dragon
early_indicators = [
    "firstblood", "firsttower", "firstherald",
    "firstmidtower", "firsttothreetowers", "firstdragon"
]

results = {}
for col in early_indicators:
    wr = df.groupby(col)["result"].mean()
    if 1.0 in wr.index:
        results[col] = wr[1.0]

ranking = pd.Series(results).sort_values(ascending=False)
print(ranking)

In [ ]:
#combined early score (0-5) - wins scale almost linearly with the score
# The numbers prove that early dominance is collectively decisive 
df["early_score"] = (
    df["firstblood"].fillna(0) +
    df["firsttower"].fillna(0) +
    df["firstherald"].fillna(0) +
    df["firstdragon"].fillna(0) +
    df["gold_lead"].fillna(0)  # already created
)

df.groupby("early_score")["result"].mean()

In [ ]:
#gold lead value decays over time - 97% before 25 miin , 58% by 35-40
bins = [0, 1500, 1800, 2100, 2400, np.inf]
labels = ["<25min", "25-30min", "30-35min", "35-40min", "40min+"]
df["length_bucket"] = pd.cut(df["gamelength"], bins=bins, labels=labels)

df.groupby(["length_bucket", "gold_lead"])["result"].mean().unstack()


In [ ]:
#kill lead at 15 translates to all structural objectives and gold 
#we see that skill in the game - getting kills, actually matters, and personal performance of a player matters > more kills > 
#more structures > more gold
df["kill_lead"] = (df["killsat15"] > df["opp_killsat15"]).astype(int)

df.groupby("kill_lead")[["firsttower", "firstmidtower", 
                          "firstherald", "firsttothreetowers",
                          "golddiffat15"]].mean()

In [ ]:
#false positive rates per indicator - First blood and first dragon are the most unreliable - also first herald by a small difference 
early_indicators = [
    "firstblood", "firsttower", "firstherald",
    "firstmidtower", "firsttothreetowers", "firstdragon",
    "firstbaron", "got_inhibitor", "gold_lead"
]

fp_results = {}
for col in early_indicators:
    # Teams that HAD the advantage
    had_advantage = df[df[col] == 1]
    # But lost
    false_positives = had_advantage[had_advantage["result"] == 0]
    
    fp_rate = len(false_positives) / len(had_advantage)
    fp_results[col] = fp_rate

fp_ranking = pd.Series(fp_results).sort_values(ascending=False)
print(fp_ranking)

In [ ]:
#false positive rates by region - PCS converts advantages most reliably 
for col in ["firstblood", "firsttower", "firstbaron", "gold_lead"]:
    print(f"\n=== {col} false positive rate by region ===")
    for region in ["LCK", "LPL", "LEC", "LCS", "PCS"]:
        sub = df_main[df_main["league"] == region]
        had = sub[sub[col] == 1]
        fp = had[had["result"] == 0]
        rate = len(fp) / len(had) if len(had) > 0 else 0
        print(f"{region}: {rate:.3f}")

In [ ]:
#false positive rate by year , sligth downward trend over time 
for col in ["firstblood", "firsttower", "firstbaron", "gold_lead"]:
    print(f"\n=== {col} false positive rate by year ===")
    for year in sorted(df["year"].unique()):
        sub = df[df["year"] == year]
        had = sub[sub[col] == 1]
        fp = had[had["result"] == 0]
        rate = len(fp) / len(had) if len(had) > 0 else 0
        print(f"{year}: {rate:.3f}")

In [ ]:
#even teams that win all 5 early indicators still lose 13.5% of the time 
for score in range(6):
    had = df[df["early_score"] == score]
    # For scores > 2, we expect wins — false positives are losses
    fp = had[had["result"] == 0]
    rate = len(fp) / len(had) if len(had) > 0 else 0
    print(f"Early score {score}/5 → lost anyway: {rate:.3f} ({len(fp)} games)")

In [ ]:
# Had gold lead AND baron but still lost
#Almost all of those happened in games over 35 minutes > coming back again to time - good factor for losing teams
fp_chain = df[
    (df["gold_lead"] == 1) &
    (df["firstbaron"] == 1) &
    (df["result"] == 0)
]

print(f"Total false positives (gold lead + baron but lost): {len(fp_chain)}")
print(f"False positive rate: {len(fp_chain) / len(df[df['gold_lead']==1][df['firstbaron']==1]):.3f}")

# Break down by region
print(fp_chain["league"].value_counts())

# Break down by year  
print(fp_chain["year"].value_counts())

# How long did these games go?
print(fp_chain["gamelength"].describe())

In [ ]:
#The most extreme false positives - Won every indicator + baron and STILL lost 
# The average game length of those is 38 minutes - confirms time is the universal explanation 
fp_full = df[
    (df["firstblood"] == 1) &
    (df["firsttower"] == 1) &
    (df["firstherald"] == 1) &
    (df["gold_lead"] == 1) &
    (df["firstbaron"] == 1) &
    (df["result"] == 0)
]

print(f"Won every early indicator + baron but still lost: {len(fp_full)} games")
print(f"\nBy region:\n{fp_full['league'].value_counts()}")
print(f"\nBy year:\n{fp_full['year'].value_counts()}")
print(f"\nGame length:\n{fp_full['gamelength'].describe()}")

In [ ]:
#will go back to xp since i never analyzed it , does having xp lead , led to baron? same with gold
#
df["xp_lead"] = (df["xpdiffat15"] > 0).astype(int)


df.groupby(["gold_lead", "xp_lead"])["firstbaron"].mean().unstack()


df.groupby(["gold_lead", "xp_lead"])["result"].mean().unstack()

In [ ]:
#cs lead generates gold so its on of the main income of gold which feeds the entire chain. 
df["cs_lead"] = (df["csdiffat15"] > 0).astype(int)
df.groupby("cs_lead")["golddiffat15"].mean()
df.groupby("cs_lead")["result"].mean()

In [ ]:
#we saw the difference with kills so lets check deaths aswell 
df["kd_ratio_15"] = df["killsat15"] / (df["deathsat15"] + 1)  # +1 avoids division by zero
df[["kd_ratio_15", "golddiffat15", "result"]].corr()

In [ ]:
#first blood has 40% false positive rate , that leads to :
df.groupby("firstblood")["golddiffat15"].mean()

#we see that the gold advantage from only a kill is quickly erased by farming

In [ ]:
# DEEPER ANALYSIS #1: Which specific combinations make up each early score level?
# We know 3/5 wins 61% but is it because of WHICH 3 indicators, not just the count?

indicators = ["firstblood", "firsttower", "firstherald", "firstdragon", "gold_lead"]

# For each meaningful score level, show every combination, count, and win rate
for score in [1, 2, 3, 4]:
    sub = df[df["early_score"] == score].copy()
    
    # Build a label showing which indicators were won
    sub["combo"] = sub[indicators].apply(
        lambda row: " + ".join([c for c in indicators if row[c] == 1]), axis=1
    )
    
    print(f"\n{'='*70}")
    print(f"EARLY SCORE {score}/5 — {len(sub)} games — overall win rate: {sub['result'].mean():.1%}")
    print(f"{'='*70}")
    
    combo_stats = sub.groupby("combo").agg(
        games=("result", "count"),
        win_rate=("result", "mean")
    ).sort_values("win_rate", ascending=False)
    
    print(combo_stats.to_string())

In [ ]:
# At each score level, compare combos WITH gold_lead vs combos WITHOUT
print("=== GOLD LEAD PREMIUM PER SCORE LEVEL ===\n")
for score in [1, 2, 3, 4]:
    sub = df[df["early_score"] == score]
    
    with_gold = sub[sub["gold_lead"] == 1]["result"].mean()
    without_gold = sub[sub["gold_lead"] == 0]["result"].mean()
    
    gap = (with_gold - without_gold) * 100
    print(f"Score {score}/5:")
    print(f"  With gold_lead:    {with_gold:.1%}")
    print(f"  Without gold_lead: {without_gold:.1%}")
    print(f"  GOLD LEAD PREMIUM: +{gap:.1f} percentage points\n")

In [ ]:
# For teams WITH a gold lead, which second objective adds the most value?
gold_only = df[(df["gold_lead"] == 1) & (df["early_score"] == 1)]["result"].mean()
print(f"Gold lead ONLY (score 1/5): {gold_only:.1%}")
print()

print("Gold lead + ONE other objective (score 2/5):")
score_2 = df[(df["gold_lead"] == 1) & (df["early_score"] == 2)]
for obj in ["firstblood", "firsttower", "firstherald", "firstdragon"]:
    combo = score_2[score_2[obj] == 1]["result"].mean()
    n = len(score_2[score_2[obj] == 1])
    print(f"  Gold lead + {obj}: {combo:.1%} ({n} games)")

In [ ]:
# How much does each objective add to a team that already has gold_lead?
print(" EACH OBJECTIVE'S MULTIPLIER EFFECT WITH GOLD LEAD \n")
gold_only_wr = df[(df["gold_lead"] == 1) & (df["early_score"] == 1)]["result"].mean()
print(f"Baseline (gold lead only): {gold_only_wr:.1%}\n")

for obj in ["firstblood", "firsttower", "firstherald", "firstdragon"]:
    # Teams with gold_lead AND this specific objective (at score 2/5)
    with_obj = df[
        (df["gold_lead"] == 1) & 
        (df[obj] == 1) & 
        (df["early_score"] == 2)
    ]["result"].mean()
    
    boost = (with_obj - gold_only_wr) * 100
    print(f"Gold lead + {obj}: {with_obj:.1%}  (boost: {boost:+.1f}pp)")

In [ ]:
# Teams WITHOUT gold lead at 15 — how often do they win and what helps?
no_gold = df[df["gold_lead"] == 0]
print(f"Total teams without gold lead: {len(no_gold)}")
print(f"Overall win rate: {no_gold['result'].mean():.1%}\n")

print("Win rate breakdown by what else they had:")
for obj in ["firstblood", "firsttower", "firstherald", "firstdragon", "firstbaron"]:
    with_obj = no_gold[no_gold[obj] == 1]["result"].mean()
    without_obj = no_gold[no_gold[obj] == 0]["result"].mean()
    print(f"  {obj}: {with_obj:.1%} WITH vs {without_obj:.1%} WITHOUT")

In [ ]:
print(" Gold lead premium across regions\n")
for region in ["LCK", "LPL", "LEC", "LCS", "PCS"]:
    sub = df_main[df_main["league"] == region]
    with_g = sub[sub["gold_lead"] == 1]["result"].mean()
    without_g = sub[sub["gold_lead"] == 0]["result"].mean()
    print(f"{region}: With gold = {with_g:.1%}, Without = {without_g:.1%}, Premium = +{(with_g-without_g)*100:.1f}pp")

print("\n Gold lead premium across years \n")
for y in sorted(df["year"].unique()):
    sub = df[df["year"] == y]
    with_g = sub[sub["gold_lead"] == 1]["result"].mean()
    without_g = sub[sub["gold_lead"] == 0]["result"].mean()
    print(f"{y}: With gold = {with_g:.1%}, Without = {without_g:.1%}, Premium = +{(with_g-without_g)*100:.1f}pp")

In [ ]:
print("=== OVERALL SIDE WIN RATE ===\n")
print(df.groupby("side")["result"].mean())
print()
print("Total games per side:")
print(df["side"].value_counts())

In [ ]:
print("SIDE'S EFFECT ON EARLY OBJECTIVES \n")
side_obj = df.groupby("side")[["firstblood", "firsttower", "firstherald", 
                                "firstdragon", "firstbaron", 
                                "gold_lead", "got_inhibitor"]].mean()
print(side_obj.T)

In [ ]:
print("=== SIDE WIN RATE BY REGION ===\n")
for region in ["LCK", "LPL", "LEC", "LCS", "PCS"]:
    sub = df_main[df_main["league"] == region]
    blue_wr = sub[sub["side"] == "Blue"]["result"].mean()
    red_wr = sub[sub["side"] == "Red"]["result"].mean()
    diff = (blue_wr - red_wr) * 100
    print(f"{region}: Blue {blue_wr:.1%} | Red {red_wr:.1%} | Blue advantage: {diff:+.1f}pp")

In [ ]:
print("=== SIDE WIN RATE BY YEAR ===\n")
for y in sorted(df["year"].unique()):
    sub = df[df["year"] == y]
    blue_wr = sub[sub["side"] == "Blue"]["result"].mean()
    red_wr = sub[sub["side"] == "Red"]["result"].mean()
    diff = (blue_wr - red_wr) * 100
    print(f"{y}: Blue {blue_wr:.1%} | Red {red_wr:.1%} | Blue advantage: {diff:+.1f}pp")

In [ ]:
print("=== GOLD LEAD VALUE BY SIDE ===\n")
for side in ["Blue", "Red"]:
    sub = df[df["side"] == side]
    with_g = sub[sub["gold_lead"] == 1]["result"].mean()
    without_g = sub[sub["gold_lead"] == 0]["result"].mean()
    print(f"{side} side: With gold = {with_g:.1%}, Without = {without_g:.1%}, Premium = +{(with_g-without_g)*100:.1f}pp")

In [ ]:
print("=== OBJECTIVE WIN RATE BY GAME LENGTH ===\n")
objectives = ["firstblood", "firsttower", "firstherald", 
              "firstdragon", "firstbaron", "got_inhibitor", "gold_lead"]

for obj in objectives:
    print(f"\n{obj}:")
    table = df.groupby(["length_bucket", obj])["result"].mean().unstack()
    print(table)

In [ ]:
print("=== DECAY PER OBJECTIVE ===\n")
print(f"{'Objective':<20} {'<25min':>8} {'35-40min':>10} {'Decay':>10}")
print("-" * 50)

for obj in ["firstblood", "firsttower", "firstherald", "firstdragon", 
            "firstbaron", "got_inhibitor", "gold_lead"]:
    short_games = df[(df["length_bucket"] == "<25min") & (df[obj] == 1)]["result"].mean()
    long_games = df[(df["length_bucket"] == "35-40min") & (df[obj] == 1)]["result"].mean()
    decay = (short_games - long_games) * 100
    print(f"{obj:<20} {short_games:>7.1%} {long_games:>9.1%} {decay:>+8.1f}pp")

In [ ]:
print("=== GOLD LEAD VS BARON DECAY COMPARISON ===\n")
print(f"{'Game Length':<15} {'Gold Lead WR':>14} {'Baron WR':>12}")
print("-" * 45)

for bucket in ["<25min", "25-30min", "30-35min", "35-40min", "40min+"]:
    sub = df[df["length_bucket"] == bucket]
    gold_wr = sub[sub["gold_lead"] == 1]["result"].mean()
    baron_wr = sub[sub["firstbaron"] == 1]["result"].mean()
    print(f"{bucket:<15} {gold_wr:>13.1%} {baron_wr:>11.1%}")

In [ ]:
print("=== OBJECTIVE SHELF LIFE (Win rate in 40+ min games) ===\n")
long_games = df[df["length_bucket"] == "40min+"]

for obj in ["firstblood", "firsttower", "firstherald", "firstdragon",
            "firstbaron", "got_inhibitor", "gold_lead"]:
    wr = long_games[long_games[obj] == 1]["result"].mean()
    n = len(long_games[long_games[obj] == 1])
    print(f"  {obj:<20} {wr:.1%}  ({n} games)")

In [ ]:
print("=== PATCH OVERVIEW ===\n")
print(f"Total unique patches: {df['patch'].nunique()}")
print(f"\nPatches per year:")
print(df.groupby("year")["patch"].nunique())

print(f"\nGame distribution per patch (top 20):")
print(df["patch"].value_counts().head(20))

In [ ]:
print("=== GOLD LEAD CORRELATION WITH WIN BY PATCH ===\n")

# Only patches with enough games to be meaningful
patch_counts = df["patch"].value_counts()
meaningful_patches = patch_counts[patch_counts >= 200].index

patch_corr = []
for patch in sorted(meaningful_patches):
    sub = df[df["patch"] == patch]
    corr = sub[["golddiffat15", "result"]].corr().iloc[0, 1]
    patch_corr.append({
        "patch": patch,
        "games": len(sub),
        "gold_correlation": corr
    })

patch_corr_df = pd.DataFrame(patch_corr).sort_values("patch")
print(patch_corr_df.to_string(index=False))

In [ ]:
print("=== BARON WIN RATE BY PATCH ===\n")

baron_by_patch = []
for patch in sorted(meaningful_patches):
    sub = df[df["patch"] == patch]
    with_baron = sub[sub["firstbaron"] == 1]["result"].mean()
    games_with_baron = len(sub[sub["firstbaron"] == 1])
    baron_by_patch.append({
        "patch": patch,
        "games_with_baron": games_with_baron,
        "win_rate": with_baron
    })

baron_df = pd.DataFrame(baron_by_patch).sort_values("patch")
print(baron_df.to_string(index=False))

In [ ]:
print("=== AVERAGE GAME LENGTH BY PATCH ===\n")

length_by_patch = df.groupby("patch").agg(
    games=("gamelength", "count"),
    avg_length_seconds=("gamelength", "mean"),
    avg_length_minutes=("gamelength", lambda x: x.mean() / 60)
).sort_index()

# Filter to meaningful patches
length_by_patch = length_by_patch[length_by_patch["games"] >= 200]
print(length_by_patch.round(2).to_string())

In [ ]:
print("=== COMEBACK RATE BY PATCH ===\n")

comeback_by_patch = []
for patch in sorted(meaningful_patches):
    sub = df[df["patch"] == patch]
    teams_ahead = sub[sub["golddiffat15"] > 1000]
    comebacks = teams_ahead[teams_ahead["result"] == 0]
    if len(teams_ahead) > 0:
        rate = len(comebacks) / len(teams_ahead)
        comeback_by_patch.append({
            "patch": patch,
            "leads_at_15": len(teams_ahead),
            "comebacks": len(comebacks),
            "comeback_rate": rate
        })

cb_df = pd.DataFrame(comeback_by_patch).sort_values("patch")
print(cb_df.to_string(index=False))

In [ ]:
# Final sanity check
print(f"Total games: {len(df)}")
print(f"\nYears: {df['year'].value_counts().sort_index().to_dict()}")
print(f"\nFirst Baron win rate: {df[df['firstbaron'] == 1]['result'].mean():.1%}")
print(f"Got Inhibitor win rate: {df[df['got_inhibitor'] == 1]['result'].mean():.1%}")
print(f"Gold lead at 15 win rate: {df[df['gold_lead'] == 1]['result'].mean():.1%}")
print(f"\nEarly score distribution:")
print(df.groupby('early_score')['result'].mean())

In [ ]:
import matplotlib.pyplot as plt

# Calculate win rates for each objective
objs = ["got_inhibitor", "firstbaron", "firsttothreetowers", "firstmidtower",
        "firsttower", "firstherald", "firstblood", "firstdragon"]
labels = ["Got Inhibitor", "First Baron", "First to 3 Towers", "First Mid Tower",
          "First Tower", "First Herald", "First Blood", "First Dragon"]
win_rates = [df[df[o] == 1]["result"].mean() * 100 for o in objs]

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#3FB950" if w >= 80 else "#0AC8B9" if w >= 70 else "#C89B3C" if w >= 60 else "#F85149" for w in win_rates]
ax.barh(labels[::-1], win_rates[::-1], color=colors[::-1], edgecolor="black")
ax.set_xlabel("Win Rate (%)")
ax.set_title("Objective Win Rate Ranking", fontsize=14, fontweight="bold")
ax.set_xlim(0, 100)
for i, v in enumerate(win_rates[::-1]):
    ax.text(v + 1, i, f"{v:.1f}%", va="center", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
score_wr = df.groupby("early_score")["result"].mean() * 100

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#F85149", "#F85149", "#C89B3C", "#0AC8B9", "#3FB950", "#3FB950"]
ax.bar([f"{int(s)}/5" for s in score_wr.index], score_wr.values, color=colors, edgecolor="black")
ax.set_ylabel("Win Rate (%)")
ax.set_xlabel("Early Score")
ax.set_title("Win Rate by Combined Early Score (0-5)", fontsize=14, fontweight="bold")
ax.set_ylim(0, 100)
for i, v in enumerate(score_wr.values):
    ax.text(i, v + 1.5, f"{v:.1f}%", ha="center", fontweight="bold")
ax.axhline(50, color="gray", linestyle="--", linewidth=1, alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
matrix = df.groupby(["gold_lead", "firstbaron"])["result"].mean().unstack() * 100

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(matrix.values, cmap="RdYlGn", vmin=0, vmax=100, aspect="auto")

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["No Baron", "Got Baron"])
ax.set_yticklabels(["Behind in Gold", "Ahead in Gold"])
ax.set_title("Win Rate: Gold Lead × Baron Interaction", fontsize=14, fontweight="bold")

for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{matrix.values[i,j]:.1f}%", ha="center", va="center",
                fontsize=20, fontweight="bold", color="white")

plt.colorbar(im, ax=ax, label="Win Rate (%)")
plt.tight_layout()
plt.show()

In [ ]:
length_order = ["<25min", "25-30min", "30-35min", "35-40min", "40min+"]
objs_to_plot = ["gold_lead", "firstbaron", "got_inhibitor", "firstherald", "firstdragon"]
labels_plot = ["Gold Lead", "First Baron", "Got Inhibitor", "First Herald", "First Dragon"]
colors_plot = ["#C89B3C", "#3FB950", "#0AC8B9", "#A371F7", "#F85149"]

fig, ax = plt.subplots(figsize=(10, 6))
for obj, lbl, col in zip(objs_to_plot, labels_plot, colors_plot):
    vals = [df[(df["length_bucket"] == b) & (df[obj] == 1)]["result"].mean() * 100 for b in length_order]
    ax.plot(length_order, vals, marker="o", linewidth=2.5, label=lbl, color=col)

ax.set_ylabel("Win Rate (%)")
ax.set_xlabel("Game Length")
ax.set_title("Objective Win Rate Decay by Game Length", fontsize=14, fontweight="bold")
ax.legend(loc="lower left")
ax.grid(alpha=0.3)
ax.set_ylim(40, 100)
plt.tight_layout()
plt.show()

In [ ]:
side_obj = df.groupby("side")[["firstblood", "firsttower", "firstherald",
                                "firstdragon", "firstbaron", "gold_lead"]].mean() * 100
labels_side = ["First Blood", "First Tower", "First Herald", "First Dragon", "First Baron", "Gold Lead"]

x = range(len(labels_side))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar([i - width/2 for i in x], side_obj.loc["Blue"].values, width, label="Blue Side", color="#3B82F6", edgecolor="black")
ax.bar([i + width/2 for i in x], side_obj.loc["Red"].values, width, label="Red Side", color="#F85149", edgecolor="black")

ax.set_ylabel("Rate of Securing Objective (%)")
ax.set_title("Blue vs Red Side — Objective Control", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(labels_side)
ax.axhline(50, color="gray", linestyle="--", linewidth=1, alpha=0.5)
ax.legend()
ax.set_ylim(0, 70)
plt.tight_layout()
plt.show()

In [ ]:
regions = ["LCK", "LPL", "LEC", "LCS", "PCS"]
premiums = []
for r in regions:
    sub = df_main[df_main["league"] == r]
    with_g = sub[sub["gold_lead"] == 1]["result"].mean() * 100
    without_g = sub[sub["gold_lead"] == 0]["result"].mean() * 100
    premiums.append(with_g - without_g)

fig, ax = plt.subplots(figsize=(9, 5))
colors_r = ["#0AC8B9" if p < 50 else "#3FB950" for p in premiums]
ax.bar(regions, premiums, color=colors_r, edgecolor="black")
ax.set_ylabel("Gold Lead Premium (pp)")
ax.set_title("Gold Lead Premium by Region", fontsize=14, fontweight="bold")
ax.set_ylim(0, 70)
for i, v in enumerate(premiums):
    ax.text(i, v + 1, f"+{v:.1f}pp", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

indicators = ["firstblood", "firsttower", "firstherald", "firstdragon", "gold_lead"]

# Build a list of all combinations and their win rates
combo_data = []
for score in [1, 2, 3, 4]:
    sub = df[df["early_score"] == score].copy()
    sub["combo"] = sub[indicators].apply(
        lambda row: " + ".join([c.replace("first", "").replace("_lead", " lead") 
                                 for c in indicators if row[c] == 1]), axis=1)
    
    combo_stats = sub.groupby("combo").agg(
        games=("result", "count"),
        win_rate=("result", "mean")
    ).reset_index()
    combo_stats["score"] = score
    combo_stats["has_gold"] = combo_stats["combo"].str.contains("gold lead")
    combo_data.append(combo_stats)

# Plot — 4 subplots (one per score level)
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, (score, data) in enumerate(zip([1, 2, 3, 4], combo_data)):
    ax = axes[i]
    data_sorted = data.sort_values("win_rate", ascending=True)
    
    # Color: green if has gold_lead, red if not
    colors = ["#3FB950" if g else "#F85149" for g in data_sorted["has_gold"]]
    
    ax.barh(data_sorted["combo"], data_sorted["win_rate"] * 100, color=colors, edgecolor="black")
    ax.set_xlabel("Win Rate (%)")
    ax.set_title(f"Score {score}/5 — All Combinations", fontsize=12, fontweight="bold")
    ax.set_xlim(0, 100)
    ax.axvline(50, color="gray", linestyle="--", linewidth=1, alpha=0.5)
    
    # Add win rate labels
    for j, v in enumerate(data_sorted["win_rate"] * 100):
        ax.text(v + 1, j, f"{v:.1f}%", va="center", fontsize=9)

# Add a legend that explains the color
fig.suptitle("Combinations by Score Level — Green = WITH gold lead, Red = WITHOUT gold lead", 
             fontsize=14, fontweight="bold", y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

years = sorted(df["year"].unique())
gold_corr = [df[df["year"] == y][["golddiffat15", "result"]].corr().iloc[0, 1] for y in years]
xp_corr = [df[df["year"] == y][["xpdiffat15", "result"]].corr().iloc[0, 1] for y in years]
cs_corr = [df[df["year"] == y][["csdiffat15", "result"]].corr().iloc[0, 1] for y in years]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(years, gold_corr, marker="o", linewidth=2.5, label="Gold Diff @15", color="#C89B3C")
ax.plot(years, xp_corr, marker="o", linewidth=2.5, label="XP Diff @15", color="#0AC8B9")
ax.plot(years, cs_corr, marker="o", linewidth=2.5, label="CS Diff @15", color="#A371F7")
ax.set_xticks(years)
ax.set_ylabel("Correlation with Win")
ax.set_xlabel("Year")
ax.set_title("Early Game Stat Correlations Across Seasons", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0, 0.6)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

indicators = ["firstdragon", "firstblood", "firstherald", "firsttower",
              "firstmidtower", "gold_lead", "firsttothreetowers", "firstbaron", "got_inhibitor"]
labels = ["First Dragon", "First Blood", "First Herald", "First Tower", "First Mid Tower",
          "Gold Lead", "First to 3 Towers", "First Baron", "Got Inhibitor"]

fp_rates = []
for col in indicators:
    had = df[df[col] == 1]
    fp = had[had["result"] == 0]
    fp_rates.append(len(fp) / len(had) * 100)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#F85149" if f > 35 else "#C89B3C" if f > 20 else "#3FB950" for f in fp_rates]
ax.barh(labels[::-1], fp_rates[::-1], color=colors[::-1], edgecolor="black")
ax.set_xlabel("False Positive Rate (%)")
ax.set_title("How Often Each Indicator FAILS to Predict a Win", fontsize=14, fontweight="bold")
ax.set_xlim(0, 50)
for i, v in enumerate(fp_rates[::-1]):
    ax.text(v + 0.5, i, f"{v:.1f}%", va="center", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
regions = ["LCK", "LPL", "LEC", "LCS", "PCS"]
rates = []
for r in regions:
    sub = df_main[df_main["league"] == r]
    teams_ahead = sub[sub["golddiffat15"] > 1000]
    comebacks = teams_ahead[teams_ahead["result"] == 0]
    rates.append(len(comebacks) / len(teams_ahead) * 100)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(regions, rates, color="#0AC8B9", edgecolor="black")
ax.set_ylabel("Comeback Rate (%)")
ax.set_title("Comeback Rate by Region (Lost despite 1k+ Gold Lead at 15)", fontsize=13, fontweight="bold")
ax.set_ylim(0, max(rates) + 1)
for i, v in enumerate(rates):
    ax.text(i, v + 0.1, f"{v:.1f}%", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()